# Dev100 — so sánh repetition_penalty 1.0 / 1.03 / 1.05

Chạy đủ 100 câu cho mỗi mức, cùng QLoRA adapter đã chọn, cùng retrieval, prompt, seed và giới hạn token của Main 2/Main 3. Chỉ thay `repetition_penalty`; không train lại, chia lại dữ liệu hoặc retrieve lại. Đây là tuning trên dev đã dùng chọn adapter, không phải kiểm định độc lập.

**Add Input bắt buộc:**
1. **Output Main 02 hoàn tất** (`legalqa_main_02_select_retrieve.ipynb`), thư mục chứa `stage2_manifest.json` có `status=complete`. Lấy `config.json`, `session.json`, `models.lock.json`, `selection.json`, `selected_adapter/`, `data/dev100.questions.json`, `data/dev100.references.json`, `data/split_manifest.json`, `dev100.retrieval.json`. Gắn toàn bộ output để kiểm tra manifest.
2. **Dataset `lighth/ver3-smoke-output`**: model weights trong `legalqa_smoke_full_v1/models/`, cùng revision Main 2 đã dùng.

**Không cần output Main 01, Main 03, Main 04; không cần dataset train.** Main 2 đã chứa adapter và split dev100 cần dùng. Chọn **GPU T4 x2 + Internet**. Notebook tự pin code đúng commit Main 2; helper thử nghiệm được nhúng riêng ngoài checkout, không sửa hash/cache nguồn.

300 lượt sinh có thể cần nhiều giờ. Ngân sách tối đa 9 giờ/phiên, có checkpoint riêng từng mức. Nếu paused, Save Output; phiên sau gắn thêm output dev100 này, đặt `PREVIOUS_OUTPUT` tới thư mục `legalqa_dev100_repetition_v1`, đồng thời giữ hai input bắt buộc trên. Không đổi cấu hình giữa các phiên.

In [ ]:
from pathlib import Path
import json, os, shutil, signal, subprocess, sys, time

SESSION_STARTED = time.monotonic()
if not Path('/kaggle').is_dir():
    raise RuntimeError('Notebook chỉ chạy trên Kaggle.')
WORK = Path('/kaggle/working')
INPUT = Path('/kaggle/input')
VERSION3_ROOT = Path('/kaggle/input/datasets/lighth/ver3-smoke-output/legalqa_smoke_full_v1')
MAIN2_OUTPUT = None  # Tự tìm đúng một stage2_manifest.json; nếu nhiều version, điền ROOT cụ thể.
PREVIOUS_OUTPUT = None  # ROOT output dev100 này từ phiên paused trước, không phải output Main 3.
WORK_HOURS = 9.0
MAX_NEW_QUESTIONS_PER_VARIANT = 0  # 0 = không giới hạn số câu ngoài ngân sách giờ; 4 để smoke rồi resume.
REPO_URL = 'https://github.com/lighth-gh/uit-dsc-2026-task2-legalqa.git'
CODE = WORK / 'legalqa_dev100_code'
RUN_ROOT = WORK / 'legalqa_dev100_repetition_v1'
MODELS = WORK / 'dev100_runtime_models'
HELPER = WORK / 'dev100_repetition.py'
PENALTIES = (1.0, 1.03, 1.05)
if not 0 < WORK_HOURS <= 9:
    raise ValueError('WORK_HOURS phải trong (0, 9].')
if not isinstance(MAX_NEW_QUESTIONS_PER_VARIANT, int) or MAX_NEW_QUESTIONS_PER_VARIANT < 0:
    raise ValueError('MAX_NEW_QUESTIONS_PER_VARIANT phải là số nguyên >= 0.')
WORK_END = SESSION_STARTED + WORK_HOURS * 3600

## Khóa code và kiểm tra input

In [ ]:
class BudgetPause(Exception):
    pass

def bounded_process(command, *, seconds=None, cwd=None, env=None):
    remaining = WORK_END - time.monotonic()
    if remaining <= 0:
        raise BudgetPause('Đã hết ngân sách phiên.')
    limit = remaining if seconds is None else min(remaining, seconds)
    print('Running:', ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(list(map(str, command)), cwd=cwd, env=env, start_new_session=True)
    try:
        rc = process.wait(timeout=limit)
    except (subprocess.TimeoutExpired, KeyboardInterrupt) as error:
        # Worker and every legalqa subprocess share this process group.
        # Stop all of them before hashing/exporting artifacts.
        try:
            os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            pass
        # The group leader may exit while a GPU child ignores SIGTERM.
        # Always kill remaining group members before exporting.
        try:
            os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        process.wait(timeout=30)
        if isinstance(error, KeyboardInterrupt):
            raise
        raise BudgetPause('Đã dừng worker theo ngân sách; tiến độ đã ghi sẽ được export.') from error
    if rc:
        raise subprocess.CalledProcessError(rc, command)


if MAIN2_OUTPUT is None:
    matches = sorted(INPUT.rglob('stage2_manifest.json'))
    if len(matches) != 1:
        raise RuntimeError(f'Cần đúng một output Main 2; tìm thấy {matches}. Điền MAIN2_OUTPUT cụ thể.')
    MAIN2_OUTPUT = matches[0].parent
MAIN2_OUTPUT = Path(MAIN2_OUTPUT)
manifest = json.loads((MAIN2_OUTPUT / 'stage2_manifest.json').read_text(encoding='utf-8'))
if manifest.get('schema') != 2 or manifest.get('stage') != 2 or manifest.get('status') != 'complete':
    raise ValueError('Cần output Main 2 schema 2 đã complete.')
PIN = manifest['code_commit']
if len(PIN) != 40 or any(c not in '0123456789abcdef' for c in PIN):
    raise ValueError('Main 2 thiếu full commit SHA hợp lệ.')
if PREVIOUS_OUTPUT is not None:
    PREVIOUS_OUTPUT = Path(PREVIOUS_OUTPUT)
    if not (PREVIOUS_OUTPUT / 'experiment.identity.json').is_file():
        raise FileNotFoundError('PREVIOUS_OUTPUT phải là ROOT của dev100 repetition experiment.')
if CODE.exists():
    remote = subprocess.check_output(['git', '-C', str(CODE), 'remote', 'get-url', 'origin'], text=True, timeout=30).strip()
    dirty = subprocess.check_output(['git', '-C', str(CODE), 'status', '--porcelain'], text=True, timeout=30).strip()
    if remote.rstrip('/') != REPO_URL.rstrip('/') or dirty:
        raise RuntimeError('Checkout không đúng origin hoặc có sửa đổi. Dùng session mới.')
else:
    bounded_process(['git', 'clone', '--no-checkout', '--depth', '1', REPO_URL, CODE], seconds=300)
bounded_process(['git', '-C', CODE, 'fetch', '--depth', '1', 'origin', PIN], seconds=300)
bounded_process(['git', '-C', CODE, 'checkout', '--detach', 'FETCH_HEAD'], seconds=60)
commit = subprocess.check_output(['git', '-C', str(CODE), 'rev-parse', 'HEAD'], text=True, timeout=30).strip()
if commit != PIN:
    raise RuntimeError('Checkout không đúng commit Main 2.')
print('Pinned Main 2 commit:', commit)
saved_models = VERSION3_ROOT / 'models'
if not (saved_models / 'models.lock.json').is_file():
    raise FileNotFoundError(saved_models / 'models.lock.json')
MODELS.mkdir(exist_ok=True)
for role in ('embedding', 'reranker', 'generator'):
    source = saved_models / role
    if not (source / 'config.json').is_file() or not list(source.glob('*.safetensors')):
        raise FileNotFoundError(f'Thiếu model weights/config Version 3: {source}')
    target = MODELS / role
    if target.exists() or target.is_symlink():
        if target.resolve() != source.resolve():
            raise RuntimeError(f'Model link khác nguồn: {target}')
    else:
        target.symlink_to(source, target_is_directory=True)
shutil.copy2(saved_models / 'models.lock.json', MODELS / 'models.lock.json')

## Helper thử nghiệm và môi trường

Helper nằm ngoài code Main 2, chỉ tạo output thí nghiệm mới. Muốn cập nhật helper trong repo, chạy `python scripts/build_dev100_notebook.py` để đồng bộ notebook. METEOR dùng scorer BTC và WordNet; lỗi kiểm tra metric sẽ dừng phiên.

In [ ]:
EXPERIMENT_HELPER = '"""Helpers embedded in dev100 notebook; execute against the Main 2 pinned checkout."""\nimport argparse\nimport copy\nimport csv\nimport io\nfrom pathlib import Path\nimport sys\n\n# The notebook writes this script outside CODE and starts it with cwd=CODE.\nsys.path.insert(0, str(Path.cwd()))\n\nfrom legalqa.generation import adapter_identity\nfrom legalqa.io import (config, copy_file, digest, file_hash, load_questions,\n                       read_json, source_hash, validate_predictions, write_json)\nfrom legalqa.metrics import compare_reports, references\nfrom legalqa.models import model_lock\nfrom legalqa.repair import repetition_ratio\nfrom legalqa.repair_v2 import loop_reasons\nfrom legalqa.retrieval import read_retrieval\nfrom legalqa.stages import generation_ready, validate_selection, verify_snapshot\n\n\nPENALTIES = (1.0, 1.03, 1.05)\n\n\ndef label(penalty):\n    return f\'rp_{penalty:.2f}\'\n\n\ndef variant_config(base, penalty):\n    if penalty not in PENALTIES:\n        raise ValueError(\'Unsupported repetition penalty\')\n    result = copy.deepcopy(base)\n    result[\'generation\'][\'repetition_penalty\'] = penalty\n    return result\n\n\ndef copy_checked(source, target):\n    target = Path(target)\n    if target.exists():\n        if file_hash(source) != file_hash(target):\n            raise ValueError(f\'Conflicting artifact; choose a new output: {target}\')\n    else:\n        copy_file(source, target)\n\n\ndef prepare(upstream, models, output, previous=None):\n    upstream, models, output = map(Path, (upstream, models, output))\n    snapshot = verify_snapshot(upstream, 2)\n    if snapshot[\'status\'] != \'complete\':\n        raise ValueError(\'Main 2 must be complete\')\n    required = {\'config.json\', \'session.json\', \'models.lock.json\', \'selection.json\',\n                \'data/dev100.questions.json\', \'data/dev100.references.json\',\n                \'data/split_manifest.json\', \'dev100.retrieval.json\',\n                \'selected_adapter/adapter_config.json\',\n                \'selected_adapter/adapter_model.safetensors\'}\n    if not required.issubset(snapshot[\'files\']):\n        raise ValueError(f\'Main 2 lacks required artifacts: {required - set(snapshot["files"])}\')\n    base = read_json(upstream / \'config.json\')\n    session = read_json(upstream / \'session.json\')\n    if source_hash() != snapshot[\'source_hash\'] or config() != base:\n        raise ValueError(\'Use the exact Main 2 pinned code/config\')\n    if session[\'config_hash\'] != digest(base):\n        raise ValueError(\'Main 2 config hash mismatch\')\n    lock = model_lock(base, models)\n    if lock != read_json(upstream / \'models.lock.json\') or lock != session[\'models\']:\n        raise ValueError(\'Version 3 model revisions differ from Main 2\')\n    questions = load_questions(upstream / \'data/dev100.questions.json\')\n    truth = references(upstream / \'data/dev100.references.json\')\n    if len(questions) != 100 or set(questions) != set(truth):\n        raise ValueError(\'Expected the same 100 dev questions and references from Main 2\')\n    selection = validate_selection(upstream)\n    read_retrieval(upstream / \'dev100.retrieval.json\', questions, base, models,\n                   expected_mode=\'full\', expected_index_hash=session[\'index_hash\'])\n    identity = {\n        \'schema\': 1, \'code_commit\': snapshot[\'code_commit\'], \'source_hash\': source_hash(),\n        \'helper_hash\': digest(Path(__file__).read_text(encoding=\'utf-8\')),\n        \'upstream_manifest_hash\': digest(snapshot), \'config\': base, \'models\': lock,\n        \'adapter\': adapter_identity(upstream / \'selected_adapter\'),\n        \'questions_hash\': digest(questions), \'reference_hash\': digest(truth),\n        \'retrieval_file_hash\': file_hash(upstream / \'dev100.retrieval.json\'),\n        \'penalties\': list(PENALTIES), \'expected_questions\': 100,\n    }\n    output.mkdir(parents=True, exist_ok=True)\n    identity_path = output / \'experiment.identity.json\'\n    if identity_path.exists() and read_json(identity_path) != identity:\n        raise ValueError(\'Existing experiment identity differs; use a new output\')\n    if previous is not None:\n        previous = Path(previous)\n        if previous.resolve() == output.resolve():\n            raise ValueError(\'Previous output must be a separate input directory\')\n        if read_json(previous / \'experiment.identity.json\') != identity:\n            raise ValueError(\'Resume identity differs; attach the same Main 2 and notebook version\')\n        # Restore only experiment files, excluding generated archives and temporary writes.\n        for path in sorted(previous.rglob(\'*\')):\n            if path.is_file() and not path.is_symlink() and path.suffix in {\'.json\', \'.jsonl\', \'.csv\', \'.txt\'}:\n                if not path.name.endswith(\'.tmp\'):\n                    copy_checked(path, output / path.relative_to(previous))\n    for relative in sorted(required - {\'session.json\'}):\n        copy_checked(upstream / relative, output / relative)\n    write_json(identity_path, identity)\n    for penalty in PENALTIES:\n        path = output / label(penalty) / \'config.json\'\n        candidate = variant_config(base, penalty)\n        if path.exists() and read_json(path) != candidate:\n            raise ValueError(f\'Variant config changed: {path}\')\n        write_json(path, candidate)\n    print(\'Reuse Main 2 dev100, retrieval and adapter:\', selection[\'label\'], flush=True)\n    print(\'Evaluation objective:\', base[\'evaluation\'], flush=True)\n    print(\'Retrieval settings (unchanged):\', base[\'retrieval\'], flush=True)\n    print(\'Generation settings (only repetition_penalty varies):\', base[\'generation\'], flush=True)\n    return identity\n\n\ndef choose(rows):\n    baseline = next(row for row in rows if row[\'repetition_penalty\'] == 1.0)\n    for row in rows:\n        row[\'delta_meteor\'] = row[\'meteor\'] - baseline[\'meteor\']\n        row[\'delta_rougeL\'] = row[\'rougeL\'] - baseline[\'rougeL\']\n        row[\'quality_preserved\'] = row[\'delta_meteor\'] >= -1e-12 and row[\'delta_rougeL\'] >= -1e-12\n        row[\'passes_screen\'] = (\n            row[\'repetition_penalty\'] != 1.0 and row[\'quality_preserved\']\n            and row[\'raw_loop_count\'] < baseline[\'raw_loop_count\']\n            and row[\'final_loop_count\'] <= baseline[\'final_loop_count\']\n            and row[\'raw_repetition_mean\'] <= baseline[\'raw_repetition_mean\'] + 1e-12\n            and row[\'final_repetition_mean\'] <= baseline[\'final_repetition_mean\'] + 1e-12)\n    candidates = [row for row in rows if row[\'passes_screen\']]\n    best = min(candidates, key=lambda row: (row[\'raw_loop_count\'], row[\'final_loop_count\'],\n                                           -row[\'meteor\'], -row[\'rougeL\'], row[\'repetition_penalty\'])) if candidates else baseline\n    return {\'recommended_penalty\': best[\'repetition_penalty\'],\n            \'improvement_found\': bool(candidates),\n            \'rule\': \'Both METEOR and ROUGE-L >= baseline; fewer raw loops; no increase in final loops or mean repetition. \'\n                    \'Rank by raw loops, final loops, METEOR, ROUGE-L, then lower penalty.\',\n            \'note\': \'Dev100 was used for adapter selection; this is tuning, not independent validation. \'\n                    \'No Main 3 configuration is changed automatically.\'}\n\n\ndef csv_report(path, rows):\n    stream = io.StringIO(newline=\'\')\n    writer = csv.DictWriter(stream, fieldnames=list(rows[0]))\n    writer.writeheader()\n    writer.writerows(rows)\n    Path(path).write_text(stream.getvalue(), encoding=\'utf-8-sig\', newline=\'\')\n\n\ndef summarize(output):\n    output = Path(output)\n    identity = read_json(output / \'experiment.identity.json\')\n    truth = references(output / \'data/dev100.references.json\')\n    if len(truth) != 100 or digest(truth) != identity[\'reference_hash\']:\n        raise ValueError(\'Comparison requires the original 100 references\')\n    rows, details, metric_identity = [], [], None\n    for penalty in PENALTIES:\n        directory = output / label(penalty)\n        prediction = directory / \'predictions.json\'\n        if not generation_ready(prediction):\n            raise ValueError(f\'Incomplete variant: {label(penalty)}; resume before comparing\')\n        pred = read_json(prediction)\n        validate_predictions(pred, truth)\n        report = read_json(directory / \'metrics.json\')\n        audit = read_json(prediction.with_suffix(\'.audit.json\'))\n        manifest = read_json(prediction.with_suffix(\'.manifest.json\'))\n        expected_config = variant_config(identity[\'config\'], penalty)\n        generated = manifest[\'identity\']\n        expected = {\'config\': expected_config, \'code\': identity[\'source_hash\'],\n                    \'adapter\': identity[\'adapter\'], \'models\': identity[\'models\'],\n                    \'questions_hash\': identity[\'questions_hash\'],\n                    \'retrieval_file_hash\': identity[\'retrieval_file_hash\'], \'mode\': \'generate\'}\n        if any(generated.get(key) != value for key, value in expected.items()):\n            raise ValueError(f\'Generation provenance mismatch: {label(penalty)}\')\n        if (report[\'samples\'] != 100 or set(report[\'per_question\']) != set(truth)\n                or report[\'reference_hash\'] != identity[\'reference_hash\']\n                or report[\'prediction_hash\'] != digest(pred)\n                or report[\'prediction_manifest\'] != manifest):\n            raise ValueError(f\'Metrics do not match this variant: {label(penalty)}\')\n        if metric_identity is not None and report[\'metric_identity\'] != metric_identity:\n            raise ValueError(\'Metric implementations differ\')\n        metric_identity = report[\'metric_identity\']\n        local = []\n        for key in truth:\n            item = audit[key]\n            raw, final = item[\'raw_answer\'], pred[key][\'answer\']\n            row = {\'id\': key, \'repetition_penalty\': penalty,\n                   \'meteor\': report[\'per_question\'][key][\'meteor\'],\n                   \'rougeL\': report[\'per_question\'][key][\'rougeL\'],\n                   \'raw_loop_reasons\': \'|\'.join(loop_reasons(raw)),\n                   \'final_loop_reasons\': \'|\'.join(loop_reasons(final)),\n                   \'raw_repetition\': repetition_ratio(raw), \'final_repetition\': repetition_ratio(final),\n                   \'hit_token_limit\': bool(item[\'hit_token_limit\']), \'route\': item[\'route\'],\n                   \'output_tokens\': item[\'output_tokens\'], \'generation_seconds\': item[\'seconds\']}\n            local.append(row)\n        details.extend(local)\n        rows.append({\'repetition_penalty\': penalty, \'samples\': 100,\n                     \'meteor\': report[\'meteor\'], \'rougeL\': report[\'rougeL\'],\n                     \'raw_loop_count\': sum(bool(r[\'raw_loop_reasons\']) for r in local),\n                     \'final_loop_count\': sum(bool(r[\'final_loop_reasons\']) for r in local),\n                     \'raw_repetition_mean\': sum(r[\'raw_repetition\'] for r in local) / 100,\n                     \'final_repetition_mean\': sum(r[\'final_repetition\'] for r in local) / 100,\n                     \'token_limit_count\': sum(r[\'hit_token_limit\'] for r in local),\n                     \'fallback_count\': sum(\'fallback\' in r[\'route\'] for r in local),\n                     \'mean_output_tokens\': sum(r[\'output_tokens\'] for r in local) / 100,\n                     \'mean_question_seconds\': sum(r[\'generation_seconds\'] for r in local) / 100})\n    selection = choose(rows)\n    for penalty in PENALTIES[1:]:\n        compare_reports(output / label(1.0) / \'metrics.json\',\n                        output / label(penalty) / \'metrics.json\',\n                        output / label(penalty) / \'paired_comparison.json\',\n                        seed=identity[\'config\'][\'seed\'])\n    write_json(output / \'comparison.json\', {\'rows\': rows, \'selection\': selection,\n        \'loop_definition\': \'legalqa.repair_v2.loop_reasons, measured on raw and final answers\',\n        \'repetition_definition\': \'duplicate fraction of lines >=35 characters; heuristic, not legal correctness\'})\n    csv_report(output / \'comparison.csv\', rows)\n    csv_report(output / \'per_question.csv\', details)\n    print(\'DEV100 REPETITION COMPARISON COMPLETE\', flush=True)\n    for row in rows:\n        print(row, flush=True)\n    print(selection, flush=True)\n    return rows, selection\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'command\', choices=[\'prepare\', \'summarize\'])\n    parser.add_argument(\'--output\', type=Path, required=True)\n    parser.add_argument(\'--upstream\', type=Path)\n    parser.add_argument(\'--models\', type=Path)\n    parser.add_argument(\'--previous\', type=Path)\n    args = parser.parse_args()\n    if args.command == \'prepare\':\n        if not args.upstream or not args.models:\n            parser.error(\'prepare requires --upstream and --models\')\n        prepare(args.upstream, args.models, args.output, args.previous)\n    else:\n        summarize(args.output)\n\n\nif __name__ == \'__main__\':\n    main()\n'
HELPER.write_text(EXPERIMENT_HELPER, encoding='utf-8')

In [ ]:
bounded_process([sys.executable, '-m', 'pip', 'install', '-q', '-r', CODE / 'requirements.txt'], seconds=1200)
bounded_process([sys.executable, '-m', 'nltk.downloader', '-q', 'wordnet', 'omw-1.4'], seconds=300)
bounded_process([sys.executable, 'scripts/check_metrics.py'], cwd=CODE, seconds=300)
prepare = [sys.executable, '-B', HELPER, 'prepare', '--upstream', MAIN2_OUTPUT,
           '--models', MODELS, '--output', RUN_ROOT]
if PREVIOUS_OUTPUT is not None:
    prepare += ['--previous', PREVIOUS_OUTPUT]
bounded_process(prepare, cwd=CODE, seconds=600)
freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True, timeout=60)
(RUN_ROOT / 'environment.freeze.txt').write_text(freeze, encoding='utf-8')

## Sinh 3 × 100 câu, chấm và so sánh

Mỗi mức sinh lại toàn bộ dev100 với cùng adapter, dùng hai GPU khi có; không lấy điểm baseline model gốc hoặc chỉ sinh lại câu lặp. `no_repeat_ngram_size`, prompt, top-k và `max_new_tokens` giữ nguyên Main 2. Log in cấu hình và số câu đã xong.

Khi hết ngân sách/cap, trạng thái `paused` là hợp lệ, chưa có kết luận chọn mức. Checkpoint riêng từng mức giữ câu đã xong. Diagnostics được đóng cả khi paused hoặc worker báo lỗi.

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

def run_variant(cfg, *args):
    env = {**os.environ, 'PYTHONUNBUFFERED': '1',
           'LEGALQA_MAX_ITEMS': str(MAX_NEW_QUESTIONS_PER_VARIANT),
           'LEGALQA_DEADLINE': str(time.time() + max(0, WORK_END - time.monotonic() - 300))}
    bounded_process([sys.executable, '-B', '-m', 'legalqa', '--config', cfg,
                     '--models', MODELS, *args], cwd=CODE, env=env)

status, failure = 'paused', None
try:
    complete = True
    for penalty in PENALTIES:
        variant = RUN_ROOT / f'rp_{penalty:.2f}'
        prediction = variant / 'predictions.json'
        print(f'DEV100 repetition_penalty={penalty}; all 100 questions; selected Main 2 adapter', flush=True)
        run_variant(variant / 'config.json', 'generate', '--multi-gpu',
                    '--questions', RUN_ROOT / 'data/dev100.questions.json',
                    '--retrieval', RUN_ROOT / 'dev100.retrieval.json',
                    '--adapter', RUN_ROOT / 'selected_adapter', '--output', prediction)
        if not all(path.is_file() for path in (prediction, prediction.with_suffix('.audit.json'),
                                               prediction.with_suffix('.manifest.json'))):
            complete = False
            print(f'Paused {variant.name}; chưa đủ 100 câu. Resume output này.', flush=True)
            break
        run_variant(variant / 'config.json', 'evaluate', '--predictions', prediction,
                    '--references', RUN_ROOT / 'data/dev100.references.json',
                    '--output', variant / 'metrics.json', '--label', variant.name)
    if complete:
        bounded_process([sys.executable, '-B', HELPER, 'summarize', '--output', RUN_ROOT], cwd=CODE, seconds=600)
        status = 'complete'
except BudgetPause as error:
    print('PAUSED:', error, flush=True)
except BaseException as error:
    status, failure = 'failed', error
finally:
    (RUN_ROOT / 'experiment.status.json').write_text(json.dumps(
        {'status': status, 'code_commit': PIN, 'penalties': PENALTIES,
         'error': str(failure) if failure else None}, ensure_ascii=False, indent=2), encoding='utf-8')
    archive_path = RUN_ROOT / 'dev100_repetition_diagnostics.zip'
    temporary = archive_path.with_suffix('.zip.tmp')
    with ZipFile(temporary, 'w', compression=ZIP_DEFLATED) as archive:
        for path in sorted(RUN_ROOT.rglob('*')):
            if path.is_file() and path.suffix in {'.json', '.jsonl', '.csv', '.txt'}:
                archive.write(path, arcname=path.relative_to(RUN_ROOT).as_posix())
    os.replace(temporary, archive_path)
    print('STATUS:', status, '| OUTPUT:', RUN_ROOT, '| DIAGNOSTICS:', archive_path, flush=True)
if failure is not None:
    raise failure

## Đọc kết quả

- `comparison.csv` / `comparison.json`: METEOR, ROUGE-L và delta so với 1.0; số câu lặp raw/final, tỷ lệ dòng lặp trung bình, số câu chạm token limit, fallback, độ dài và thời gian trung bình/câu (không phải tổng thời gian hai GPU).
- `per_question.csv`: 300 dòng để so sánh từng ID; lý do phát hiện lặp. Xem `rp_*/predictions.audit.json` để đọc `raw_answer` và `rp_*/predictions.json` để đọc đáp án cuối.
- `rp_1.03/paired_comparison.json`, `rp_1.05/paired_comparison.json`: paired bootstrap METEOR/ROUGE-L so với 1.0; không bảo đảm chất lượng trên private.
- Chỉ đề xuất mức mới khi **cả METEOR và ROUGE-L không giảm**, số câu raw bị lặp giảm, số câu final bị lặp và tỷ lệ dòng lặp trung bình không tăng. Trong các mức đạt, ưu tiên ít câu lặp raw/final hơn, rồi METEOR và ROUGE-L cao hơn. Nếu không mức nào đạt, giữ **1.0**, `improvement_found=false`.
- Bộ phát hiện lặp là heuristic, không đánh giá tính đúng pháp lý. Notebook không thay config Main 3 và không tạo submission private.

Đủ điều kiện hoàn tất khi log có `DEV100 REPETITION COMPARISON COMPLETE` và `STATUS: complete`, đủ 100 câu ở cả ba mức. Nếu `paused`, gắn toàn bộ output để resume; diagnostics ZIP không chứa adapter weights nên không thay thế output đầy đủ khi resume.